In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

c:\Coding\Python\internship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Shail Patel\AppData\Local\Temp\ipykernel_27552\4025367806.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [12]:
from dotenv import load_dotenv
load_dotenv()

True

### Step 1A - Ingestion (Document Ingestion)

In [29]:
# video_id = "J5_-l7WIO_w"    # from https://www.youtube.com/watch?v=J5_-l7WIO_w&list=PLKnIA16_RmvaTbihpo4MtzVm4XOQa0ER0&index=19
video_id = "Fa_V9fP2tpU"    # from https://www.youtube.com/watch?v=Fa_V9fP2tpU

try:
    api_client = YouTubeTranscriptApi()
    transcript_list = api_client.list(video_id)
    transcript_obj = transcript_list.find_transcript(['en'])    #'hi' for hindi
    data_blocks = transcript_obj.fetch()
    
    transcript = " ".join(chunk.text for chunk in data_blocks)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions are available for this video.")
except NoTranscriptFound:
    print("No English captions found for this video.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

here a list of all basic machine learning terms in 22 minutes artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence this can include understanding language recognizing images solving problems or making decisions AI aims to mimic human cognitive functions through various techniques including machine learning but not all AI is machine learning for example rule-based systems can use predefined logical rules to analyze medical data and provide diagnostic recommendations without needing to learn from data patterns typical chess playing engines would be considered AI but not machine learning because they follow specific rules in search algorithms and don't always learn from data machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly programmed for each task in machine learning algorithms identify patterns and re

### Step 1B - Indexing (Text Splitting)

In [30]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
chunks

[Document(metadata={}, page_content="here a list of all basic machine learning terms in 22 minutes artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence this can include understanding language recognizing images solving problems or making decisions AI aims to mimic human cognitive functions through various techniques including machine learning but not all AI is machine learning for example rule-based systems can use predefined logical rules to analyze medical data and provide diagnostic recommendations without needing to learn from data patterns typical chess playing engines would be considered AI but not machine learning because they follow specific rules in search algorithms and don't always learn from data machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly programmed for each task in machine learnin

### Step 1C & 1D - Indexing (Embedding Generation and Storing in Vector Store)

In [31]:
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')
vector_store = FAISS.from_documents(chunks, embeddings)

In [33]:
vector_store.index_to_docstore_id

{0: '82503947-0fa8-47a1-91b6-e24cb935b672',
 1: '57afa880-f804-4035-859d-80f1c4beb088',
 2: '46aef88a-1bdc-421e-b7b0-04cf0995b03c',
 3: 'aee2a160-b3ea-4e00-bcf5-e38d56157617',
 4: '59243528-7bc9-4f4b-8a2f-bc7b8401ef61',
 5: '862dee89-64a8-4f5b-ab78-b0b4c115d024',
 6: 'ac63793b-fa25-4d63-b738-b7dc6a580d41',
 7: 'c8ff39ca-052c-4ce8-a1a5-5023cf651132',
 8: '3b967e99-492a-447f-a334-dbc9e01aa15d',
 9: '37964316-f208-4418-b281-6a432f9ad415',
 10: 'cc7cb0ca-1345-4fbe-9579-3f3d70b83518',
 11: 'acaca5a8-d209-4d84-9d97-39a7af5fb3e6',
 12: '850d0139-8094-4799-8a78-4a3106ae4406',
 13: '3343c9e1-f3d1-48ab-9901-200981fcad5f',
 14: '540f6186-2daa-4b7f-874c-6517b6911c37',
 15: 'f393e502-593c-4a62-86e4-bf9f566e39fc',
 16: '777d25ec-a4c3-4177-992a-a68a7ddb8454',
 17: 'acdf720c-b3de-4b1e-9950-b647b7478237',
 18: '6258e920-7c33-4a4b-b0ef-0023fa3862c3',
 19: '6776379c-3de2-4c6b-9d5b-5c3a9ae6c399',
 20: '9d777d18-0e5f-4d5a-8316-473b497c9725',
 21: '53981a4e-f280-4554-8523-f7b269650f2d',
 22: '03673e37-9ccc-

In [36]:
vector_store.get_by_ids(['d62602ee-1341-4791-afcc-90903716ef45'])

[Document(id='d62602ee-1341-4791-afcc-90903716ef45', metadata={}, page_content='helpful share it with someone who you think might also like it and get started on one of the tutorials in the description or on this very Channel also consider liking the video and subscribing to be notified about similar content in the future thanks for watching')]

### Step 2 - Retrieval

In [37]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})  # 4 similar vectors
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E419B82610>, search_kwargs={'k': 4})

Topics in the video:

Artificial Intelligence (AI), Machine Learning, Algorithm, Data, Model, Model fitting, Training Data, Test Data, Supervised Learning, Unsupervised Learning, Reinforcement Learning, Feature, Feature engineering, Feature Scaling, Dimensionality, Target, Instance, Label, Model complexity, Bias & Variance, Bias Variance Tradeoff, Noise, Overfitting & Underfitting, Validation & Cross Validation, Regularization, Batch, Epoch, Iteration, Parameter, Hyperparameter, Cost Function, Gradient Descent, Learning Rate, Evaluation

In [39]:
retriever.invoke("What is the difference between AI and ML")

[Document(id='82503947-0fa8-47a1-91b6-e24cb935b672', metadata={}, page_content="here a list of all basic machine learning terms in 22 minutes artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence this can include understanding language recognizing images solving problems or making decisions AI aims to mimic human cognitive functions through various techniques including machine learning but not all AI is machine learning for example rule-based systems can use predefined logical rules to analyze medical data and provide diagnostic recommendations without needing to learn from data patterns typical chess playing engines would be considered AI but not machine learning because they follow specific rules in search algorithms and don't always learn from data machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly 

### Step 3 - Augmentation

In [40]:
llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=0.2)

In [49]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.
        
        {context}
        Question: {question}
    """,
    input_variables=['context', 'question']
)

In [57]:
question = "What is the difference between AI and ML"
retrieved_docs = retriever.invoke(question)

In [58]:
retrieved_docs

[Document(id='82503947-0fa8-47a1-91b6-e24cb935b672', metadata={}, page_content="here a list of all basic machine learning terms in 22 minutes artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence this can include understanding language recognizing images solving problems or making decisions AI aims to mimic human cognitive functions through various techniques including machine learning but not all AI is machine learning for example rule-based systems can use predefined logical rules to analyze medical data and provide diagnostic recommendations without needing to learn from data patterns typical chess playing engines would be considered AI but not machine learning because they follow specific rules in search algorithms and don't always learn from data machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly 

In [59]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)  # Merge into one string
context_text

"here a list of all basic machine learning terms in 22 minutes artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence this can include understanding language recognizing images solving problems or making decisions AI aims to mimic human cognitive functions through various techniques including machine learning but not all AI is machine learning for example rule-based systems can use predefined logical rules to analyze medical data and provide diagnostic recommendations without needing to learn from data patterns typical chess playing engines would be considered AI but not machine learning because they follow specific rules in search algorithms and don't always learn from data machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly programmed for each task in machine learning algorithms identify patterns and\n

In [60]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

### Step 4 - Generation

In [61]:
answer = llm.invoke(final_prompt)
print(answer.text)

Artificial intelligence refers to the capability of machines to perform tasks that typically require human intelligence, such as understanding language, recognizing images, solving problems, or making decisions. Machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks over time without being explicitly programmed for each task. Not all AI is machine learning; for example, rule-based systems and typical chess-playing engines are considered AI but not machine learning because they follow predefined logical rules or search algorithms rather than learning from data patterns.


### Building a Chain

In [77]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [70]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [80]:
parallel_chain.invoke("What is Feature Engineering")

{'context': "model essentially it's any measurable property that helps the model make predictions for example in a house price prediction model features might include the square footage number of bedrooms location and age of the house for an email spam detector features could include the number of capitalized words the number of URLs in the text or whether the sender is in your contacts the selection and Engineering of relevant features sometimes called feature extraction or feature design is often crucial to a model success as they need to capture the important aspects of the data that relate to the prediction task feature engineering is the process of creating new more informative features from existing raw data to improve a model's performance feature engineering involves using domain knowledge and creativity to transform or combine original features into more meaningful ones for for example instead of just using raw date values you might create features like day of the week or is h

In [81]:
parser = StrOutputParser()

In [82]:
main_chain = parallel_chain | prompt | llm | parser

In [83]:
main_chain.invoke("can you summarize the video")

'The video provides an overview of basic machine learning terms. Key points include:\n\n*   **Artificial Intelligence vs. Machine Learning:** AI is the broad capability of machines to mimic human intelligence (such as solving problems or recognizing images). Machine learning is a specific branch of AI that enables computers to learn from data patterns rather than relying solely on predefined rules.\n*   **Model Evaluation:** This is the process of measuring a model\'s performance on unseen data to ensure it has learned patterns rather than just memorizing training data. Metrics vary by task, such as accuracy, precision, recall, and F1 score for classification, or mean squared error and R-squared for regression.\n*   **Gradient Descent:** This is a process used to minimize model error by adjusting parameters based on the gradient. It uses a learning rate to determine step size. The video also discusses "momentum-based gradient descent," which helps models avoid getting stuck in local mi